# 一、前言

这节主要是知道什么是卷积， 卷积层怎么算，核是什么与padding（填充） 和 stride（步幅） 的作用，并用具体代码例程验证

# 二、笔记

四个符号（全章通用）
| 符号 | 含义 | 举例 |
|---|---|---|
| **n** | 输入的尺寸（高 n_h、宽 n_w，可不相等） | 图像 8×8 → n_h=8, n_w=8 |
| **k** | 卷积核的尺寸（高 k_h、宽 k_w） | 核 3×3 → k_h=3, k_w=3 |
| **p** | 填充量（PyTorch 的 padding 参数，**每边**补几圈 0） | padding=1 → 四周各补 1 圈 |
| **s** | 步幅（窗口每次跳几格，PyTorch 的 stride） | stride=1 逐格滑，stride=2 跳着滑 |

# 1.卷积的作用

参数爆炸 → 两个常识 → 卷积层诞生<br>

全书把 kernel 叫"卷积核"，把"互相关"叫"卷积"

第一步，看问题有多严重：<br>
全连接：100 万像素 → 1000 个隐藏单元，一层就要 10⁶×10³ = 10⁹ 参数。<br>
然后：1000×1000 图像 → 1000×1000 隐藏表示，全连接要 10¹² 参数，10¹² 个权重，训练难度非常大。

第二步，动脑子：图像和表格不一样。表格里特征没有结构，MLP 是对的；但图像有结构——一只猫在图片左上角和右下角，本质是同一只猫。于是两个常识登场：<br>
平移不变性：检测物体和它出现在哪无关 → 输出位置 (i,j) 的权重不该依赖 (i,j) 本身 → 四阶张量 W[i,j,a,b] 坍缩成 V[a,b]，位置索引 (i,j) 直接被扔掉了。<br>
局部性：判断一个像素像不像边缘，看它周围几格就够，不用看整张图 → a,b 只在 ±Δ 范围内非零 → 窗口有限。

第三步，看效果：参数从 10¹² 砍到 (2Δ+1)² 个——一个 3×3 核就 9 个权重。省了 11 个数量级。代价是"归纳偏置"：模型默认特征只平移、只局部。图像符合这个先验，所以 CNN 又小又准。

第四步，通道：图像是 RGB 三维的 → 输入变三维 → 每个位置要一组特征 → V 变成 [V]_{a,b,c,d}。

代价：模型天生只认"平移+局部"的模式——这叫归纳偏置。<br>
CNN 依靠局部感受野和权值共享两个假设减少参数量：局部感受野让神经元只连接局部像素而非全部像素；权值共享让同一个卷积核在全图滑动时复用一套权重。同等输入输出下，全连接参数量可达千万级别，卷积层通常仅为千级别，参数量远小于全连接网络。

# 2.卷积怎么算，核是什么

互相关运算：核在输入上从左到右、从上到下滑动，每个位置算"窗口内逐元素相乘再求和"。就这一件事。核"放得下"的位置才输出，所以尺寸是 (n−k+1)²。<br>
如：
| 输入X  |  核K |
|---|---|
|0  1  2 | 0  1|
|3  4  5 | 2  3|
|6  7  8 |     |

核放在输入左上角，盖住 0,1,3,4 → 逐元素相乘再求和：<br>
0×0 + 1×1 + 3×2 + 4×3 = 0 + 1 + 6 + 12 = 19   ← 输出的第一个数<br>
然后核往右滑 1 格（s=1）盖住 1,2,4,5 再算，一行算完换下一行，直到盖不住为止。<br>
输出多大？公式：输出 = n − k + 1（高、宽各算一遍）<br>
这里 n=3, k=2 → 高 = 3−2+1 = 2，宽同样 2 → 输出 2×2，共 4 个数：19, 25, 37, 43<br>
为什么是 n−k+1：核"放得下"的位置才输出。3 格长的一维里放 2 格长的核，只有 2 个位置（盖 0-1、盖 1-2）

一个核 = 一个特征检测器：核 [1,−1] 检测垂直边缘——相邻两像素相同输出 0，白→黑输出 1，黑→白输出 −1。把输入转置后输出全 0，说明它只认垂直方向。这就是为什么后面需要几十个核（多通道）各管一个方向。

核是学出来的（本节最重要的信念）：给你 X 和 Y，用梯度下降学核，10 轮 loss 从 6.4 降到 0.02，学出 ≈[1,−1]。这和学权重本质完全一样——特征不用人设计，训练自己会找。这就是 CNN 的核心信念。

# 3.两个旋钮

为什么要管尺寸：连续卷积会让图越来越小——240×240 过 10 层 5×5 卷积变 200×200，边界信息全丢（英文版说"削掉 30% 的图像"）；反过来，有时又想让图快速变小省算力。

padding（填充）= 保尺寸：边界补 0。配方 p = k−1 时输出和输入同尺寸；核用奇数（3/5/7）就是为了两边均匀补。

stride（步幅）：窗口跳着滑。s=2 尺寸减半，可以替代一层池化做下采样。

卷积输出尺寸标准公式：<br>
$\boldsymbol{O=\left\lfloor \frac{n+2p-k}{s}\right\rfloor+1}$<br>
n：输入边长<br>
p：单边填充 padding<br>
k：卷积核大小<br>
s：步长 stride<br>
$\lfloor\ \rfloor$：向下取整<br>
PyTorch 代码里 padding=1 指的是每边补 1 圈（总共 2）<br>
注意 PyTorch 默认：Conv2d 是 stride=1、padding=0；MaxPool2d 是 stride=kernel_size

问：为什么 k=3 的核，p=1 就能保住尺寸？（提示：核伸出输入几格？每边各补回几格？）<br>
 3×3 卷积核滑动到边缘时，核会向图像外侧伸出 1 格；padding=1 就是图像的每一侧都补上 1 格像素，刚好把伸出去的那一块补齐，所以输出尺寸和输入尺寸保持不变。

# 二、代码例程

## 1.手写互相关（核心）

In [13]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y
# 默认没有填充，步幅为 1

#.......验证.................

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

# 这个函数是纯张量运算，autograd 会自动记录梯度——这正是后面"学习卷积核"能跑的前提。

tensor([[19., 25.],
        [37., 43.]])

## 2.自定义二维卷积层

In [15]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1)) 

    def forward(self, x):
        return crr2d(x, self.weight) + self.bias

为什么 weight 用 rand、bias 用 zeros：权重随机打破对称性，偏置从 0 开始无害。torch.zeros(1)，生成含有 1 个元素、值为 0 的一维张量

为什么用 nn.Parameter 包一层：不包的话 optimizer 不知道它是可学习参数，net.parameters() 里没有它，训练等于没训练。

为什么 forward 里能直接用 corr2d：它内部只有张量运算，在 autograd 的图里，梯度能穿过它流到 weight 上。

## 3.边缘检测（理解卷积核的"语义"）

In [16]:
X = torch.ones((6, 8))  # 6 行，8 列矩阵，全部是 1（白色区域）
X[:, 2:6] = 0   # 所有行的第 2~5 列赋值为 0（黑色）
print(X)
K = torch.tensor([[1.0, -1.0]])  # 行向量核 `[1, -1]`，左边像素 − 右边像素
Y = corr2d(X, K)
print(Y)
# 期望：每行中间出现 1（白→黑）和 -1（黑→白）

corr2d(X.t(), K)   # 转置后全是 0 → 该核只检测垂直边缘

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])
tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

**垂直边缘（竖直边缘）**：图像里，**像素值突然发生变化的一条竖线**。
线条是从上往下延伸的。

为什么 K=[[1,-1]] 能检测垂直边缘：相邻两像素相等→和为 0；白(1)到黑(0)→1；黑到白→−1。核就是"相邻像素之差"。<br>
输出形状：8-2+1=7，输出为 6×7<br>
- 滑到第 1、2 列：1-0=1→ 检测到白→黑垂直边缘<br>
- 滑到第 5、6 列：0-1=-1 → 检测到黑→白垂直边缘<br>
- 其余位置两边像素相等：1-1=0,0-0=0

corr2d(X.t(), K)<br>
X.t() 是矩阵转置，把原先垂直边缘 → 变成水平边缘<br>
图像变成 6 列，8 行，黑白分界线横着走。

为什么转置后输出全 0：因为核还是 [1, -1]，它只横向比较左右两个像素，看不出上下的变化，所以结果全 0。核只在宽度方向差分，垂直方向的颜色突变它看不到。核是"定向"的——这是后面多通道存在的理由之一：一层要很多核才能覆盖各种方向。<br>
核 [[1,-1]]：专门检测垂直边缘，不能检测水平边缘<br>
如果想检测水平边缘，需要换成竖直卷积核<br>
K = torch.tensor([[1.0],
                  [-1.0]])

## 4.从数据学习卷积核（重点）

In [17]:
# 定义卷积层
conv2d = nn.Conv2d(1, 1, kernel_size = (1, 2), bias = False)

# 改变形状适配 Conv2d 输入格式
X = X.reshape((1, 1, 6, 8))  # batch‑size 为 1；通道数为 1；图像高度为 6；图像宽度为 8
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X)          # 前向传播：卷积预测
    l = (Y_hat - Y) ** 2       # 逐元素平方损失
    conv2d.zero_grad()         # 梯度清零
    l.sum().backward()         # 反向传播求梯度
    conv2d.weight.data[:] -= lr * conv2d.weight.grad   # SGD 更新权重
    if(i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {l.sum():.3f}')

# 查看学到的卷积核
conv2d.weight.data.reshape((1, 2))

epoch 2, loss 11.911
epoch 4, loss 2.502
epoch 6, loss 0.626
epoch 8, loss 0.190
epoch 10, loss 0.066


tensor([[ 1.0087, -0.9587]])

nn.Conv2d(in_channels, out_channels, kernel_size, bias)<br>
in_channels=1：输入通道数 1（灰度单通道图片）<br>
out_channels=1：输出通道数 1，输出一张特征图<br>
kernel_size=(1,2)：卷积核尺寸：高 = 1，宽 = 2，就是一行两列 `[a, b]`，正好匹配之前的 `[1, -1]`<br>
bias=False：关掉偏置<br>
nn.Conv2d 是 PyTorch **二维卷积层（2‑D 卷积）**，对图像 / 特征图做**互相关滑动窗口运算**，用来提取图像特征：边缘、纹理、角点、高级图案<br>
Conv2d 做的是**互相关，不是数学上翻转核的卷积**

为什么 bias=False：目标 Y 恰好无偏置，多一个参数只会让核学不干净，验证"核本身"被学对。

为什么 zero_grad()：不清梯度，10 轮会累积出 10 份梯度，一步就飞。

为什么是 weight.data[:] -= ...：原地修改 data，绕开 autograd 对新权重的追踪；直接 weight -= ... 会污染计算图。

学出来的核 ≈ [1, −1]——训练闭环在这里第一次作用在"看不见的核"上，这是 CNN 的核心信念：特征不用手设计，学出来。

## 5. 填充

In [18]:
def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)   # 批量=1, 通道=1，拼接得到元组 `(1, 1, 8, 8)`
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])     # 丢掉批量、通道，把 (1, 1, 8, 8) → 变回二维 (8, 8)
    
conv2d = nn.Conv2d(1, 1, kernel_size = 3, padding = 1)
X = torch.rand(size = (8, 8))   # 从区间 [0，1) 均匀分布随机采样，生成一个 8 行、8 列 的二维张量
comp_conv2d(conv2d, X).shape          # torch.Size([8, 8])

torch.Size([8, 8])

函数 comp_conv2d 的作用
一个**包装工具函数**，专门解决痛点：
nn.Conv2d 要求输入是 4 维 (batch, channel, H, W)，但你手上只有二维图像矩阵 (H,W)。
这个函数自动补上前两维，跑完卷积，再把多余两维删掉，还给你二维结果。

为什么 reshape 成 (1,1,h,w)：nn.Conv2d 要求 4 维输入（批量, 通道, 高, 宽），这是 PyTorch 的约定。不 reshape 直接报错。

为什么算完丢掉前两维：我们只关心空间尺寸变化。

为什么 pad=1, k=3 → 尺寸不变：公式 (n_h−k_h+p_h+1) = 8−3+2+1 = 8。p = k−1 是"same padding"的通用配方，记住它，LeNet 第一层就是 padding=2, k=5。

## 6.非对称核与步幅

In [28]:
# 非对称核 + 非对称填充：k=(5,3), p=(2,1) → 仍保持 8×8
conv2d = nn.Conv2d(1, 1, kernel_size = (5, 3), padding = (2, 1))
comp_conv2d(conv2d, X).shape      # torch.Size([8, 8])

# 步幅 2 → 尺寸减半
conv2d = nn.Conv2d(1, 1, kernel_size = 3, padding = 1, stride = 2)
comp_conv2d(conv2d, X).shape      # torch.Size([4, 4])

# 综合：k=(3,5), p=(0,1), s=(3,4)
conv2d = nn.Conv2d(1, 1, kernel_size = (3, 5), padding = (0,1), stride = (3, 4))
comp_conv2d(conv2d, X).shape      # torch.Size([2, 2])


torch.Size([2, 2])

输出尺寸公式（垂直方向 / 高度、水平方向 / 宽度分开算）<br>
$H_{out}=\left\lfloor \frac{H_{in}+2\cdot p_h - k_h}{s_h}\right\rfloor+1$<br>
$W_{out}=\left\lfloor \frac{W_{in}+2\cdot p_w - k_w}{s_w}\right\rfloor+1$<br>
输入图像：$H_{in}=8,\;W_{in}=8$

为什么步幅能减半尺寸：窗口每次跳 s 格，能放的窗口数变少。实践中 stride=2 常用于替代一层池化做下采样。

# 三、测试小结

输入 8×8、k=3、p=1、s=2，输出尺寸？用公式算。
4×4

输入 8×8、k=(3,5)、p=(0,1)、s=(3,4)，输出尺寸？
2×2

为什么 corr2d 输出是 (n−k+1)？如果 k 比输入还大，会发生什么？
核每滑 1 格算 1 个位置，最后一个能放下的起点是 n−k（从 0 数起），所以一共能放 n−k+1 个位置——"能放下的窗口数"就是输出长度；如果卷积核尺寸 大于 输入图像，那么则窗口一次都放不下，没有任何可以滑动的有效位置。

核 [[1,−1]] 检测垂直边缘的原理？把它转置（[[1],[−1]]）后能检测什么？
核 [[1,−1]] 可以让左右两列像素值相减，如果两列值相同，则得 0，如不相同则为 1，那么则会有像素点的变化，这些点组成一条竖线段；检测水平边缘

cell 从数据学习卷积核 里为什么必须 l.sum().backward() 而不能直接 l.backward()？为什么每轮要 zero_grad()？
因为 .backward() 只处理标量，只能对标量求导；如果不清零本轮梯度会加到上一轮旧梯度上面导致梯度数值错误，参数更新跑偏。

"p=k−1 使输出和输入同尺寸"。用公式验证 k=5、p=2、s=1 的情形。
输入 H_in=8，H_out={8+2*2 -5}/1 + 1=8，得证

为什么 CNN 核尺寸几乎总是奇数（1/3/5/7）？和"以像素为中心"有什么关系？（提示：p=(k−1)/2 整除）
k 奇数，p 得到整数，上下、左右填充数量相等，卷积窗口滑动时，卷积核有一个精确的中心点像素，核可以以当前像素为中心向四周扩展；k 偶数，(k−1)/2 得到小数，无法做到上下填充数 = 左右填充数，找不到一个正中间像素，中心点落在 4 个像素缝隙之间。